# OCR recognition fine-tuning smoke test (Colab)


## 1. GPU и установка (только для обучения §6–§10)

In [ ]:
!nvidia-smi

In [1]:
%pip uninstall -y opencv-python-headless opencv-python opencv-contrib-python numpy
%pip install -q "numpy==1.26.4" "opencv-python==4.6.0.66" "opencv-contrib-python==4.6.0.66"
%pip install -q "pillow==11.1.0" "pyyaml>=6.0" "pandas==2.2.2"
%pip uninstall -y paddlepaddle paddlepaddle-gpu
%pip install paddlepaddle-gpu==2.6.2
print("Готово. Runtime → Restart session, затем ячейка 2.")

Found existing installation: opencv-python-headless 4.13.0.92
Uninstalling opencv-python-headless-4.13.0.92:
  Successfully uninstalled opencv-python-headless-4.13.0.92
Found existing installation: opencv-python 4.13.0.92
Uninstalling opencv-python-4.13.0.92:
  Successfully uninstalled opencv-python-4.13.0.92
Found existing installation: opencv-contrib-python 4.13.0.92
Uninstalling opencv-contrib-python-4.13.0.92:
  Successfully uninstalled opencv-contrib-python-4.13.0.92
Found existing installation: numpy 2.0.2
Uninstalling numpy-2.0.2:
  Successfully uninstalled numpy-2.0.2
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 125.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 MB 44.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.1/67.1 MB 39.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are ins

In [1]:
import paddle
import numpy as np
import cv2

print(f"✓ PaddlePaddle: {paddle.__version__}")
print(f"✓ CUDA: {paddle.is_compiled_with_cuda()}")
if paddle.is_compiled_with_cuda():
    print(f"✓ GPU: {paddle.device.cuda.get_device_name(0)}")
print(f"✓ NumPy: {np.__version__}")
print(f"✓ OpenCV: {cv2.__version__}")

try:
    import paddleocr
    print(f"PaddleOCR: {paddleocr.__version__}")
except:
    print("PaddleOCR установлен из исходников")


✓ PaddlePaddle: 2.6.2
✓ CUDA: True
✓ GPU: Tesla T4
✓ NumPy: 2.5.0
✓ OpenCV: 4.13.0
PaddleOCR установлен из исходников


In [ ]:
# import sys
# sys.path.insert(0, '/content/PaddleOCR')

## 2. Импорты и поиск ручной разметки

In [2]:
import json
import random
import re
import shutil
import subprocess
import sys
import tarfile
import urllib.request
from pathlib import Path

import pandas as pd
from google.colab import drive

try:
    import paddle
    _paddle_version = paddle.__version__
except ImportError:
    paddle = None
    _paddle_version = 'not installed'

def fix_train_deps() -> None:
    """Re-pin numpy/opencv for PaddleOCR tools/infer."""
    cmd = (
        'pip uninstall -y opencv-python-headless opencv-python opencv-contrib-python numpy && '
        'pip install -q "numpy==1.26.4" "opencv-python==4.6.0.66" "opencv-contrib-python==4.6.0.66"'
    )
    subprocess.run(cmd, shell=True, check=True)


def ensure_paddle_gpu() -> None:
    global paddle, _paddle_version
    try:
        import paddle as _paddle
        if tuple(int(x) for x in _paddle.__version__.split('.')[:2]) >= (3, 0):
            paddle = _paddle
            _paddle_version = _paddle.__version__
            return
    except ImportError:
        pass
    print('Ставлю paddlepaddle-gpu 3.2.2...')
    subprocess.run(
        'pip uninstall -y paddlepaddle paddlepaddle-gpu && '
        'pip install -q "paddlepaddle-gpu==3.2.2" '
        '-i https://www.paddlepaddle.org.cn/packages/stable/cu126/',
        shell=True,
        check=True,
    )
    import paddle as _paddle
    paddle = _paddle
    _paddle_version = _paddle.__version__


def verify_train_deps() -> None:
    code = 'import numpy as np; import cv2; print("OK", np.__version__, cv2.__version__)'
    result = subprocess.run([sys.executable, '-c', code], capture_output=True, text=True)
    print(result.stdout.strip() or result.stderr.strip())
    if result.returncode != 0:
        raise RuntimeError('numpy/opencv не подходят — §11 вызовет fix_train_deps(); для обучения нужна §1 + restart.')

drive.mount('/content/drive')

REC_MODEL_NAME = 'cyrillic_PP-OCRv5_mobile_rec'
PROJECT_ROOT = Path('/content')
DRIVE_ROOT = Path('/content/drive/MyDrive/Avar OCR ANN')
PADDLEOCR_DIR = PROJECT_ROOT / 'PaddleOCR'
# print(PADDLEOCR_DIR)
DATASET_DIR = PROJECT_ROOT / 'dataset' / 'avar_rec_manual'
OUTPUT_DIR = PROJECT_ROOT / 'output' / 'rec_avar_v5'
INFER_DIR = PROJECT_ROOT / 'output' / 'cyrillic_PP-OCRv5_mobile_rec_avar_infer'

for path in [DATASET_DIR, OUTPUT_DIR, INFER_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print('model:', REC_MODEL_NAME)
print('paddle:', _paddle_version)
if paddle is not None:
    print('cuda compiled:', paddle.is_compiled_with_cuda())
    print('gpu count:', paddle.device.cuda.device_count())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
model: cyrillic_PP-OCRv5_mobile_rec
paddle: 2.6.2
cuda compiled: True
gpu count: 1


In [3]:
manual_files = sorted((DRIVE_ROOT / 'classical_layout').rglob('manual_corrections.md'))
if not manual_files:
    raise FileNotFoundError('Не нашла manual_corrections.md в Drive/classical_layout')

for i, path in enumerate(manual_files):
    print(i, path)

MANUAL_INDEX = 0
manual_md_path = manual_files[MANUAL_INDEX]
page_dir = manual_md_path.parent
ocr_json_path = page_dir / 'ocr_results.json'

print('manual:', manual_md_path)
print('page_dir:', page_dir)
print('ocr json:', ocr_json_path, ocr_json_path.exists())

0 /content/drive/MyDrive/Avar OCR ANN/classical_layout/0_tlyarata_6-7_na_pechat_page_001/connected_components/manual_corrections.md
1 /content/drive/MyDrive/Avar OCR ANN/classical_layout/0_tlyarata_6-7_na_pechat_page_002/connected_components/manual_corrections.md
2 /content/drive/MyDrive/Avar OCR ANN/classical_layout/0_tlyarata_6-7_na_pechat_page_003/connected_components/manual_corrections.md
3 /content/drive/MyDrive/Avar OCR ANN/classical_layout/0_tlyarata_6-7_na_pechat_page_004/connected_components/manual_corrections.md
4 /content/drive/MyDrive/Avar OCR ANN/classical_layout/0_tlyarata_6-7_na_pechat_page_005/connected_components/manual_corrections.md
5 /content/drive/MyDrive/Avar OCR ANN/classical_layout/0_tlyarata_6-7_na_pechat_page_006/connected_components/manual_corrections.md
6 /content/drive/MyDrive/Avar OCR ANN/classical_layout/0_tlyarata_8_-na_pechat_page_003/connected_components/manual_corrections.md
manual: /content/drive/MyDrive/Avar OCR ANN/classical_layout/0_tlyarata_6-7_n

## 3. Парсинг `manual_corrections.md` и сбор dataset

Берём только блок `Corrected`. Каждый crop становится одной OCR recognition training sample.

In [4]:
# Если файл существует — возвращает его.
# Иначе: ищет все manual_corrections.md в classical_layout,
# выводит список и возвращает первый

def refresh_manual_path(current_path: Path) -> Path:
    """Re-resolve manual_corrections.md in case Google Drive state changed."""
    if current_path.exists():
        return current_path

    print('manual_md_path не найден, ищу заново в Drive...')
    refreshed = sorted((DRIVE_ROOT / 'classical_layout').rglob('manual_corrections.md'))
    if refreshed:
        for i, path in enumerate(refreshed):
            print(i, path)
        return refreshed[0]

    raise FileNotFoundError(
        f'Не найден manual_corrections.md. Проверьте, что файл есть на Drive и Drive смонтирован: {current_path}'
    )

# ищет исправленные результаты распознавания
def parse_manual_corrections(md_path: Path) -> list[dict]:
    md_path = refresh_manual_path(md_path)
    # print('Reading:', md_path)
    # print('Exists:', md_path.exists())

    text = md_path.read_text(encoding='utf-8')
    pattern = re.compile(
        r'## block_(\d+).*?Crop: `([^`]+)`.*?Corrected:\s*```text\n(.*?)\n```',
        re.S,
    )
    records = []
    for match in pattern.finditer(text):
        order = int(match.group(1))
        crop_rel = match.group(2).strip()
        corrected = match.group(3).strip()
        if not corrected:
            continue

        crop_path = md_path.parent / crop_rel
        if not crop_path.exists():
            # Fallback: if metadata was moved, resolve by crop filename in local crops folder.
            crop_path = md_path.parent / 'crops' / Path(crop_rel).name

        if not crop_path.exists():
            print('skip missing crop:', crop_path)
            continue

        records.append({
            'order': order,
            'crop_path': crop_path,
            'corrected_text': corrected,
        })
    return sorted(records, key=lambda r: r['order'])

# # пример запуска
# manual_md_path = refresh_manual_path(manual_md_path)
# page_dir = manual_md_path.parent
# ocr_json_path = page_dir / 'ocr_results.json'

# records = parse_manual_corrections(manual_md_path)
# print('records:', len(records))
# pd.DataFrame([{'order': r['order'], 'crop': r['crop_path'].name, 'text': r['corrected_text'][:80]} for r in records]).head(10)

In [9]:
import random
from pathlib import Path
import shutil
from collections import Counter

random.seed(42)

# очистить старый датасет
if DATASET_DIR.exists():
    shutil.rmtree(DATASET_DIR)
(DATASET_DIR / 'images').mkdir(parents=True, exist_ok=True)

# все записи из всех manual_corrections.md
all_items = []
for md_path in manual_files:
    records = parse_manual_corrections(md_path)
    for rec in records:
        all_items.append((str(rec['crop_path']), rec['corrected_text']))

print(f'Всего записей (с дубликатами): {len(all_items)}')

# удалить дубликаты по пути кропа
seen = set()
unique = []
for path, text in all_items:
    if path not in seen:
        seen.add(path)
        unique.append((path, text))

print(f'Уникальных: {len(unique)}')

# скопировать имена кропов и сделать относительные пути
label_entries = []
for abs_path, text in unique:
    src = Path(abs_path)
    # уникальное имя: страница_блок.png
    page_name = src.parent.parent.parent.name
    crop_name = src.name
    dst_name = f"{page_name}_{crop_name}"
    dst = DATASET_DIR / 'images' / dst_name
    if not dst.exists():
        shutil.copy2(src, dst)
    rel = dst.relative_to(DATASET_DIR).as_posix()
    label_entries.append((rel, text))

# перемешать → разделить на выборки
random.shuffle(label_entries)
split = int(len(label_entries) * 0.8)
train_entries = label_entries[:split]
val_entries = label_entries[split:]

def write_list(path, entries):
    lines = []
    for rel, text in entries:
        clean_text = text.replace('\n', ' ').replace('\r', ' ') # переносы строк внутри текста
        clean_text = ' '.join(clean_text.split()) # убрать двойные пробелы
        lines.append(f'{rel}\t{clean_text}')
    path.write_text('\n'.join(lines) + '\n', encoding='utf-8')

write_list(DATASET_DIR / 'train_list.txt', train_entries)
write_list(DATASET_DIR / 'val_list.txt', val_entries)
write_list(DATASET_DIR / 'all_list.txt', label_entries)

print(f'\nTrain: {len(train_entries)}')
print(f'Val: {len(val_entries)}')
print(f'Всего: {len(label_entries)}')

# поиск дубликатов
all_files = [l.split('\t')[0] for l in open(str(DATASET_DIR / 'all_list.txt'))]
dupes = [f for f, c in Counter(all_files).items() if c > 1]
print(f'Дубликатов: {len(dupes)}') # типа название газеты

Всего записей (с дубликатами): 143
Уникальных: 143

Train: 114
Val: 29
Всего: 143
Дубликатов: 0


In [8]:
# label_entries

## 4. Качество исходной OCR-модели до fine-tuning

Сравниваем исходный `ocr_results.json` с ручным `Corrected` и считаем:

- `CER` — character error rate;
- `WER` — word error rate.

Это baseline до обучения. Если `ocr_results.json` получен моделью `cyrillic_PP-OCRv5_mobile_rec`, метрика относится именно к ней.

In [12]:
# Расстояние Левенштейна
def edit_distance(a: list, b: list) -> int:
    dp = list(range(len(b) + 1))
    for i, ca in enumerate(a, start=1):
        prev, dp[0] = dp[0], i
        for j, cb in enumerate(b, start=1):
            cur = dp[j]
            dp[j] = min(dp[j] + 1, dp[j - 1] + 1, prev + (ca != cb))
            prev = cur
    return dp[-1]

# Character Error Rate: edit_distance (символы) / длина эталона
def cer(pred: str, gt: str) -> float:
    return edit_distance(list(pred), list(gt)) / max(1, len(gt))

# Word  Error Rate: edit_distance (слова) / длина эталона
def wer(pred: str, gt: str) -> float:
    return edit_distance(pred.split(), gt.split()) / max(1, len(gt.split()))

In [29]:
# результаты OCR дефолтной модели
baseline_texts = {}
for page_dir in (DRIVE_ROOT / 'classical_layout').iterdir():
    if not page_dir.is_dir():
        continue
    json_path = page_dir / 'connected_components' / 'ocr_results.json'
    if not json_path.exists():
        continue
    data = json.loads(json_path.read_text(encoding='utf-8'))
    items = data.get('results', data) if isinstance(data, dict) else data
    for item in items:
        order = item.get('order', item.get('block', 0))
        text = item.get('text', '') or ''
        # Ключ: страница_block_NNN.png
        key = f"{page_dir.name}_block_{int(order):03d}.png"
        baseline_texts[key] = text
# сравнить с эталонами
rows = []
for list_name in ['train_list.txt', 'val_list.txt']:
    list_path = DATASET_DIR / list_name
    if not list_path.exists():
        continue
    for line in list_path.read_text(encoding='utf-8').splitlines():
        if '\t' not in line:
            continue
        rel, gt = line.split('\t', 1)
        # извлечь имя файла из пути
        fname = rel.split('/')[-1]  # images/xxx_block_000.png -> xxx_block_000.png
        pred = baseline_texts.get(fname, '')
        rows.append({
            'file': rel,
            'pred': pred[:80],
            'gt': gt[:80],
            'cer': cer(pred, gt),
            'wer': wer(pred, gt),
            'match': pred == gt,
        })

baseline_df = pd.DataFrame(rows)
print(f'Всего примеров: {len(baseline_df)}')
print(f'Mean CER: {baseline_df["cer"].mean():.4f} ({baseline_df["cer"].mean()*100:.1f}%)')
print(f'Mean WER: {baseline_df["wer"].mean():.4f} ({baseline_df["wer"].mean()*100:.1f}%)')
print(f'Exact match: {baseline_df["match"].mean():.4f} ({baseline_df["match"].mean()*100:.1f}%)') # идеальных строк
# display(df.head(100))

Всего примеров: 143
Mean CER: 0.0718 (7.2%)
Mean WER: 0.1865 (18.7%)
Exact match: 0.3497 (35.0%)


In [18]:
baseline_metrics_path = DATASET_DIR / 'baseline_metrics.json'
baseline_metrics = {
    'manual_corrections': str(manual_md_path),
    'ocr_results': str(ocr_json_path),
    'mean_cer': float(baseline_df['cer'].mean()),
    'mean_wer': float(baseline_df['wer'].mean()),
    'exact_match': float((baseline_df['pred'] == baseline_df['gt']).mean()),
    'items': rows,
}
baseline_metrics_path.write_text(json.dumps(baseline_metrics, ensure_ascii=False, indent=2), encoding='utf-8')
print('Saved:', baseline_metrics_path)

Saved: /content/dataset/avar_rec_manual/baseline_metrics.json


## 5. Character dictionary

Словарь для обучения = `ppocrv5_cyrillic_dict.txt` + дополнительные символы из разметки (например `№`, `*`).

In [19]:
# label_entries из файлов датасета
label_entries = []
for list_name in ['train_list.txt', 'val_list.txt']:
    list_path = DATASET_DIR / list_name
    if list_path.exists():
        for line in list_path.read_text(encoding='utf-8').splitlines():
            if '\t' in line:
                rel, text = line.split('\t', 1)
                label_entries.append((rel, text, 0))  # order не важен для словаря

In [20]:
chars = set()
for _, text, _ in label_entries:
    chars.update(text)

char_dict_path = DATASET_DIR / 'char_dict_avar.txt'
char_dict_path.write_text('\n'.join(sorted(chars)) + '\n', encoding='utf-8')
print('char dict:', char_dict_path)
print('chars:', len(chars))

char dict: /content/dataset/avar_rec_manual/char_dict_avar.txt
chars: 114


## 6. PaddleOCR training repo

In [21]:
import os
from pathlib import Path

os.chdir('/content')

# удалить старую версию PaddleOCR
if PADDLEOCR_DIR.exists():
    shutil.rmtree(PADDLEOCR_DIR)

# клонировать main
!git clone --depth 1 -b main https://github.com/PaddlePaddle/PaddleOCR.git {PADDLEOCR_DIR}

# конфиг для выбранной модели
expected_cfg = PADDLEOCR_DIR / 'configs' / 'rec' / 'PP-OCRv5' / 'multi_language' / 'cyrillic_PP-OCRv5_mobile_rec.yml'

if not expected_cfg.exists():
    # если не нашли, где ожидали
    configs = list(PADDLEOCR_DIR.rglob('*cyrillic_PP-OCRv5*'))
    if configs:
        expected_cfg = configs[0]
        print(f"Найден: {expected_cfg}")
    else:
        raise FileNotFoundError("Конфиг cyrillic_PP-OCRv5_mobile_rec не найден")

# зависимости из репо
os.chdir(str(PADDLEOCR_DIR))
!pip install -q -r requirements.txt

import paddle

print('PaddleOCR OK:', PADDLEOCR_DIR)
print('paddle:', paddle.__version__, 'cuda:', paddle.is_compiled_with_cuda())
print('Конфиг:', expected_cfg)

Cloning into '/content/PaddleOCR'...
remote: Enumerating objects: 2910, done.
remote: Counting objects: 100% (2910/2910), done.
remote: Compressing objects: 100% (2077/2077), done.
remote: Total 2910 (delta 783), reused 2580 (delta 740), pack-reused 0 (from 0)
Receiving objects: 100% (2910/2910), 96.56 MiB | 33.89 MiB/s, done.
Resolving deltas: 100% (783/783), done.
Найден: /content/PaddleOCR/configs/rec/PP-OCRv5/multi_language/cyrillic_PP-OCRv5_mobile_rec.yaml
PaddleOCR OK: /content/PaddleOCR
paddle: 2.6.2 cuda: True
Конфиг: /content/PaddleOCR/configs/rec/PP-OCRv5/multi_language/cyrillic_PP-OCRv5_mobile_rec.yaml


## 7. Pretrained recognition weights

Скачиваем train-веса **`cyrillic_PP-OCRv5_mobile_rec_pretrained.pdparams`** — ту же модель, что используется в `ocr_on_classical_crops.ipynb`.

In [22]:
pretrained_dir = PROJECT_ROOT / 'pretrained' / REC_MODEL_NAME
pretrained_dir.mkdir(parents=True, exist_ok=True)
pretrained_path = pretrained_dir / 'cyrillic_PP-OCRv5_mobile_rec_pretrained.pdparams'

if not pretrained_path.exists():
    url = 'https://paddle-model-ecology.bj.bcebos.com/paddlex/official_pretrained_model/cyrillic_PP-OCRv5_mobile_rec_pretrained.pdparams'
    print('Downloading pretrained...')
    urllib.request.urlretrieve(url, pretrained_path)

print('pretrained:', pretrained_path, pretrained_path.exists())

pretrained: /content/pretrained/cyrillic_PP-OCRv5_mobile_rec/cyrillic_PP-OCRv5_mobile_rec_pretrained.pdparams True


In [ ]:
# # правильные numpy и opencv
# !pip install -q --force-reinstall "numpy==1.26.4" "opencv-python==4.6.0.66" "opencv-contrib-python==4.6.0.66"

# import paddle
# print('PaddleOCR OK:', PADDLEOCR_DIR)
# print('paddle:', paddle.__version__, 'cuda:', paddle.is_compiled_with_cuda())
# print('Конфиг:', expected_cfg)

## 8. Training config

Шаблон: `configs/rec/PP-OCRv5/multi_language/cyrillic_PP-OCRv5_mobile_rec.yaml`.

In [39]:
src_cfg = PADDLEOCR_DIR / 'configs' / 'rec' / 'PP-OCRv5' / 'multi_language' / 'cyrillic_PP-OCRv5_mobile_rec.yaml'
if not src_cfg.exists():
    src_cfg = PADDLEOCR_DIR / 'configs' / 'rec' / 'PP-OCRv5' / 'multi_language' / 'cyrillic_PP-OCRv5_mobile_rec.yml'

# базовый словарь без изменений
base_dict = PADDLEOCR_DIR / 'ppocr' / 'utils' / 'dict' / 'ppocrv5_cyrillic_dict.txt'
train_dict_path = DATASET_DIR / 'ppocrv5_cyrillic_dict.txt'
shutil.copy2(base_dict, train_dict_path)

print(f'Словарь: {len(open(train_dict_path).readlines())} символов (без изменений)')

# создание кастомного конфига
config_text = src_cfg.read_text(encoding='utf-8')
config_text = config_text.replace(
    'character_dict_path: ./ppocr/utils/dict/ppocrv5_cyrillic_dict.txt',
    f'character_dict_path: {train_dict_path}',
)
config_text = config_text.replace('max_text_length: &max_text_length 25', 'max_text_length: &max_text_length 120') # дляна текста одного кропа
config_text = config_text.replace('distributed: true', 'distributed: false')
config_text = config_text.replace('num_workers: 8', 'num_workers: 0') # один GPU
config_text = config_text.replace('num_workers: 4', 'num_workers: 0') # чтобы запускалось в Colab
config_text = config_text.replace('prob: 0.5', 'prob: 0.3') # аугментация

# config_text = config_text.replace('image_shape: [3, 48, 320]', 'image_shape: [3, 48, 640]')
# config_text = config_text.replace('image_shape: [48, 320, 3]', 'image_shape: [48, 640, 3]')

config_path = PADDLEOCR_DIR / 'configs' / 'rec_avar_v5.yaml'
config_path.write_text(config_text, encoding='utf-8')

print(f'✓ Конфиг: {config_path}')

Словарь: 850 символов (без изменений)
✓ Конфиг: /content/PaddleOCR/configs/rec_avar_v5.yaml


In [69]:
# # проверить конфиг
# !grep max_text_length /content/PaddleOCR/configs/rec_avar_v5.yaml
# !grep name: /content/PaddleOCR/configs/rec_avar_v5.yaml | head -5

In [71]:
# # проверить словарь
# # должно быть 850 строк
# !wc -l /content/dataset/avar_rec_manual/ppocrv5_cyrillic_dict.txt
# # веса должны быть 852
# import paddle
# state = paddle.load('/content/pretrained/cyrillic_PP-OCRv5_mobile_rec/cyrillic_PP-OCRv5_mobile_rec_pretrained.pdparams')
# for k, v in state.items():
#     if 'ctc_head.fc' in k:
#         print(f'{k}: {v.shape}')

## 9. Training

`EPOCH_NUM = 1` только для проверки, что код обучения

Для реального обучения 30-50.

#### с логами

In [40]:
import os
import sys

os.environ['LD_LIBRARY_PATH'] = '/usr/local/lib/python3.12/dist-packages/paddle/libs:' + os.environ.get('LD_LIBRARY_PATH', '')

os.chdir(str(PADDLEOCR_DIR))
sys.path.insert(0, str(PADDLEOCR_DIR / 'tools'))

# Только патч get_logger
import tools.program as program_module
from ppocr.utils.logging import get_logger as _original_get_logger

def patched_get_logger(log_file=None, **kwargs):
    return _original_get_logger(log_file=log_file)

program_module.get_logger = patched_get_logger

# sys.argv
train_list = DATASET_DIR / 'train_list.txt'
val_list = DATASET_DIR / 'val_list.txt'

EPOCH_NUM = 50
train_n = sum(1 for _ in train_list.open(encoding='utf-8'))
BATCH_SIZE = max(1, min(8, train_n))

sys.argv = [
    'train.py',
    '-c', 'configs/rec_avar_v5.yaml',
    '-o',
    f'Global.pretrained_model={pretrained_path}',
    f'Global.save_model_dir={OUTPUT_DIR}',
    f'Global.epoch_num={EPOCH_NUM}',
    f'Train.dataset.data_dir={DATASET_DIR}',
    f'Train.dataset.label_file_list=["{train_list}"]',
    f'Eval.dataset.data_dir={DATASET_DIR}',
    f'Eval.dataset.label_file_list=["{val_list}"]',
    f'Train.sampler.first_bs={BATCH_SIZE}',
    f'Train.loader.batch_size_per_card={BATCH_SIZE}',
    f'Eval.loader.batch_size_per_card={BATCH_SIZE}',
    'Train.loader.drop_last=false',
    'Eval.loader.drop_last=false',
]

print(f'Train: {train_n}, Batch: {BATCH_SIZE}')
print('='*60)

import paddle
import random
import numpy as np

config, device, logger, vdl_writer = program_module.preprocess(is_train=True)

seed = config["Global"].get("seed", 1024)
random.seed(seed)
np.random.seed(seed)
paddle.seed(seed)

import importlib.util
spec = importlib.util.spec_from_file_location("train", str(PADDLEOCR_DIR / 'tools' / 'train.py'))
train_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(train_module)

print('\nЗапуск main()...\n')
train_module.main(config, device, logger, vdl_writer)

print('\n' + '='*60)
!ls -la {OUTPUT_DIR}/

Train: 114, Batch: 8
[2026/06/27 13:51:37] ppocr INFO: Architecture : 
[2026/06/27 13:51:37] ppocr INFO:     Backbone : 
[2026/06/27 13:51:37] ppocr INFO:         name : PPLCNetV3
[2026/06/27 13:51:37] ppocr INFO:         scale : 0.95
[2026/06/27 13:51:37] ppocr INFO:     Head : 
[2026/06/27 13:51:37] ppocr INFO:         head_list : 
[2026/06/27 13:51:37] ppocr INFO:             CTCHead : 
[2026/06/27 13:51:37] ppocr INFO:                 Head : 
[2026/06/27 13:51:37] ppocr INFO:                     fc_decay : 1e-05
[2026/06/27 13:51:37] ppocr INFO:                 Neck : 
[2026/06/27 13:51:37] ppocr INFO:                     depth : 2
[2026/06/27 13:51:37] ppocr INFO:                     dims : 120
[2026/06/27 13:51:37] ppocr INFO:                     hidden_dims : 120
[2026/06/27 13:51:37] ppocr INFO:                     kernel_size : [1, 3]
[2026/06/27 13:51:37] ppocr INFO:                     name : svtr
[2026/06/27 13:51:37] ppocr INFO:                     use_guide : True
[2026/0

eval model:: 100%|██████████| 4/4 [00:00<00:00, 15.43it/s]

[2026/06/27 13:57:50] ppocr INFO: cur metric, acc: 0.4482757074911354, norm_edit_dis: 0.7287205341659061, fps: 144.0928374609959


[2026/06/27 13:57:51] ppocr INFO: save best model is to /content/output/rec_avar_v5/best_accuracy
[2026/06/27 13:57:51] ppocr INFO: best metric, acc: 0.4482757074911354, is_float16: False, norm_edit_dis: 0.7287205341659061, fps: 144.0928374609959, best_epoch: 48
[2026/06/27 13:57:53] ppocr INFO: epoch: [48/50], global_step: 1008, lr: 0.000027, acc: 0.749998, norm_edit_dis: 0.925649, CTCLoss: 0.732281, NRTRLoss: 1.013371, loss: 1.749926, avg_reader_cost: 0.03620 s, avg_batch_cost: 0.19990 s, avg_samples: 4.6, ips: 23.01099 samples/s, eta: 0:00:14, max_mem_reserved: 1946 MB, max_mem_allocated: 1793 MB
[2026/06/27 13:57:54] ppocr INFO: save model in /content/output/rec_avar_v5/latest
[2026/06/27 13:57:55] ppocr INFO: epoch: [49/50], global_step: 1010, lr: 0.000027, acc: 0.687499, norm_edit_dis: 0.912009, CTCLoss: 0.600662, NRTRLoss: 1.013371, loss: 1.619834, avg_reader_cost: 0.05388 s, avg_batch_cost: 0.09930 s, avg_samples: 1.6, ips: 16.11253 samples/s, eta: 0:00:13, max_mem_reserved: 19

#### то же самое без логов

In [ ]:
import os
import sys

os.environ['LD_LIBRARY_PATH'] = '/usr/local/lib/python3.12/dist-packages/paddle/libs:' + os.environ.get('LD_LIBRARY_PATH', '')

os.chdir(str(PADDLEOCR_DIR))
sys.path.insert(0, str(PADDLEOCR_DIR / 'tools'))

# Патч get_logger
import tools.program as program_module
from ppocr.utils.logging import get_logger as _original_get_logger
import logging

def patched_get_logger(log_file=None, **kwargs):
    logger = _original_get_logger(log_file=log_file)
    # Показываем только WARNING и выше, убираем INFO
    logger.setLevel(logging.WARNING)
    return logger

program_module.get_logger = patched_get_logger

# sys.argv
train_list = DATASET_DIR / 'train_list.txt'
val_list = DATASET_DIR / 'val_list.txt'

EPOCH_NUM = 50
train_n = sum(1 for _ in train_list.open(encoding='utf-8'))
BATCH_SIZE = max(1, min(8, train_n))

sys.argv = [
    'train.py',
    '-c', 'configs/rec_avar_v5.yaml',
    '-o',
    f'Global.pretrained_model={pretrained_path}',
    f'Global.save_model_dir={OUTPUT_DIR}',
    f'Global.epoch_num={EPOCH_NUM}',
    f'Train.dataset.data_dir={DATASET_DIR}',
    f'Train.dataset.label_file_list=["{train_list}"]',
    f'Eval.dataset.data_dir={DATASET_DIR}',
    f'Eval.dataset.label_file_list=["{val_list}"]',
    f'Train.sampler.first_bs={BATCH_SIZE}',
    f'Train.loader.batch_size_per_card={BATCH_SIZE}',
    f'Eval.loader.batch_size_per_card={BATCH_SIZE}',
    'Train.loader.drop_last=false',
    'Eval.loader.drop_last=false',
]

print(f'Train: {train_n}, Val: {sum(1 for _ in open(str(val_list)))}, Epochs: {EPOCH_NUM}')
print('='*60)

import paddle, random, numpy as np

config, device, logger, vdl_writer = program_module.preprocess(is_train=True)

seed = config["Global"].get("seed", 1024)
random.seed(seed)
np.random.seed(seed)
paddle.seed(seed)

import importlib.util
spec = importlib.util.spec_from_file_location("train", str(PADDLEOCR_DIR / 'tools' / 'train.py'))
train_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(train_module)

# Перенаправляем вывод train.py — только строки с 'epoch:'
import io, sys as _sys
old_stdout = _sys.stdout
_sys.stdout = io.StringIO()

train_module.main(config, device, logger, vdl_writer)

output = _sys.stdout.getvalue()
_sys.stdout = old_stdout

# Показываем только строки с метриками
for line in output.split('\n'):
    if 'epoch:' in line or 'save model' in line or 'best metric' in line:
        # Обрезаем длинные строки
        if 'epoch:' in line:
            parts = line.split(',')
            short = ','.join(parts[:3] + parts[-2:])  # epoch, step, lr + ips, eta
            print(short.strip())
        else:
            print(line.strip())

!ls -la {OUTPUT_DIR}/ | tail -5

Train: 16, Val: 4, Epochs: 50

-rw-r--r-- 1 root root       116 Jun 26 18:38 iter_epoch_50.states
-rw-r--r-- 1 root root 127251950 Jun 26 18:38 latest.pdopt
-rw-r--r-- 1 root root  71544379 Jun 26 18:38 latest.pdparams
-rw-r--r-- 1 root root        99 Jun 26 18:38 latest.states
-rw-r--r-- 1 root root     62480 Jun 26 18:21 train.log


## 10. Export inference model + save to Drive

После обучения экспортируем модель в формат, который понимает `PaddleOCR(text_recognition_model_dir=...)`.

In [41]:
import os, sys, shutil, yaml
os.chdir(str(PADDLEOCR_DIR))
sys.path.insert(0, str(PADDLEOCR_DIR / 'tools'))

import paddle
from ppocr.modeling.architectures import build_model

# лучший чекпоинт
best_ckpt = OUTPUT_DIR / 'best_accuracy'
DICT_PATH = DATASET_DIR / 'ppocrv5_cyrillic_dict.txt'

if INFER_DIR.exists():
    shutil.rmtree(INFER_DIR)
INFER_DIR.mkdir(parents=True, exist_ok=True)

with open(PADDLEOCR_DIR / 'configs' / 'rec_avar_v5.yaml', 'r') as f:
    config = yaml.safe_load(f)

arch_config = config['Architecture']
arch_config['Head']['out_channels_list'] = {'CTCLabelDecode': 852, 'NRTRLabelDecode': 856}

model = build_model(arch_config)
state_dict = paddle.load(str(best_ckpt) + '.pdparams')
model.set_state_dict(state_dict)
model.eval()

# экспорт с фиксированным размером
input_spec = paddle.static.InputSpec(shape=[None, 3, 48, 320], dtype='float32', name='x')
# input_spec = paddle.static.InputSpec(shape=[None, 3, 48, 640], dtype='float32', name='x') # не вытягивать по вертикали
# input_spec = paddle.static.InputSpec(shape=[None, 3, 48, 960], dtype='float32', name='x') # не вытягивать по вертикали
model = paddle.jit.to_static(model, input_spec=[input_spec])
paddle.jit.save(model, str(INFER_DIR / 'inference'))

# inference.yml
chars = [ln.strip() for ln in DICT_PATH.read_text(encoding='utf-8').splitlines() if ln.strip()]
inference_yml = {
    'Global': {'model_name': REC_MODEL_NAME},
    'PostProcess': {'name': 'CTCLabelDecode', 'character_dict': chars}
}
with open(INFER_DIR / 'inference.yml', 'w', encoding='utf-8') as f:
    yaml.dump(inference_yml, f, allow_unicode=True, default_flow_style=False)

# путь на диске
DRIVE_OUT = DRIVE_ROOT / 'models' / REC_MODEL_NAME
if DRIVE_OUT.exists():
    shutil.rmtree(DRIVE_OUT)
shutil.copytree(INFER_DIR, DRIVE_OUT)
print(f'Модель на Drive: {DRIVE_OUT}')

Модель на Drive: /content/drive/MyDrive/Avar OCR ANN/models/cyrillic_PP-OCRv5_mobile_rec


## 11. Проверка кастомной модели (inference)


In [42]:

import sys, json
sys.path.insert(0, str(PADDLEOCR_DIR))
os.chdir(str(PADDLEOCR_DIR))

import yaml, numpy as np, paddle
from ppocr.data import create_operators, transform
from ppocr.postprocess.rec_postprocess import CTCLabelDecode
from ppocr.modeling.architectures import build_model

DICT_PATH = DATASET_DIR / 'ppocrv5_cyrillic_dict.txt'
CKPT_PATH = OUTPUT_DIR / 'best_accuracy'

post_process = CTCLabelDecode(character_dict_path=str(DICT_PATH), use_space_char=True)

with open(PADDLEOCR_DIR / 'configs' / 'rec_avar_v5.yaml', 'r') as f:
    config = yaml.safe_load(f)

transforms = []
for op in config['Eval']['dataset']['transforms']:
    op_name = list(op)[0]
    if 'Label' in op_name: continue
    if op_name == 'RecResizeImg':
        op = {op_name: dict(op[op_name])}
        op[op_name]['infer_mode'] = True
    elif op_name == 'KeepKeys':
        op = {op_name: {'keep_keys': ['image']}}
    transforms.append(op)
ops = create_operators(transforms, config['Global'])

arch_config = config['Architecture']
arch_config['Head']['out_channels_list'] = {'CTCLabelDecode': 852, 'NRTRLabelDecode': 856}

model = build_model(arch_config)
state = paddle.load(str(CKPT_PATH) + '.pdparams')
model.set_state_dict(state)
model.eval()
paddle.set_device('gpu')
print('Модель загружена\n')

def normalize_label(text):
    return ' '.join(text.replace('\t', ' ').replace('\r', ' ').split()).strip()

LIST_FILE = DATASET_DIR / 'val_list.txt'
# LIST_FILE = DATASET_DIR / 'train_list.txt'
items = []
lines = [l for l in LIST_FILE.read_text(encoding='utf-8').splitlines() if l.strip() and '\t' in l]
for idx, line in enumerate(lines):
    rel, gt = line.split('\t', 1)
    gt = normalize_label(gt)
    img_path = str(DATASET_DIR / rel)
    pred = ''

    if Path(img_path).exists():
        with open(img_path, 'rb') as f:
            batch = transform({'image': f.read()}, ops)
        images = paddle.to_tensor(np.expand_dims(batch[0], axis=0))

        with paddle.no_grad():
            preds = model(images)
        if isinstance(preds, dict):
            preds = preds['ctc']

        result = post_process(preds)
        if result and result[0]:
            pred = normalize_label(str(result[0][0]))

    match = pred == gt
    items.append({'file': rel, 'gt': gt[:80], 'pred': pred[:80], 'match': match})

import pandas as pd
df = pd.DataFrame(items)
print(f'\nТочность: {df["match"].mean():.2%}')
display(df)

Модель загружена


Точность: 24.14%


,file,gt,pred,match
0,images/0_tlyarata_6-7_na_pechat_page_005_block...,межрайонная туберкулезная больни- ца». При изу...,ва Iзаман,False
1,images/0_tlyarata_6-7_na_pechat_page_001_block...,САПАРАЛ ВА CYАЛАЛ ЦIумаз бусен лъураб борхалъи...,вва цIияман,False
2,images/0_tlyarata_6-7_na_pechat_page_006_block...,Федералияб централъ регионал кӀодо гьарулел руго,бедералб централ регинал то гьрлеруо,False
3,images/0_tlyarata_6-7_na_pechat_page_005_block...,ЛЪАРАТIА,ЛЪАРАТIА,True
4,images/0_tlyarata_8_-na_pechat_page_003_block_...,Макьуялъул тадбир,Макьуялъултадбир,False
5,images/0_tlyarata_6-7_na_pechat_page_004_block...,Апрелалъул 27 абилеб къоялда ЛъаратӀа культури...,в IебеIеаI Iтизмлн,False
6,images/0_tlyarata_8_-na_pechat_page_003_block_...,Сахлъи цIуниялъе сабабаздасан ккола макьуялъул...,вм,False
7,images/0_tlyarata_8_-na_pechat_page_003_block_...,Радал кьижун ватидал ибну ГIаббасица (р.гI.) ж...,вCм,False
8,images/0_tlyarata_6-7_na_pechat_page_003_block...,ТIокIлъаби,ТIокIлъаби,True
9,images/0_tlyarata_6-7_na_pechat_page_003_block...,"гьабуна школалъул, росдал адми- нистрациялъул ...",вма,False


In [30]:
# Проверить размеры "плохих" изображений
import cv2
for idx, row in df.iterrows():
    if row['pred'] in ('C', '', 'C и'):
        img = cv2.imread(str(DATASET_DIR / row['file']))
        if img is not None:
            h, w = img.shape[:2]
            ratio = w / h if h > 0 else 0
            print(f"{row['file'].split('/')[-1]}: {w}x{h}, ratio={ratio:.1f}, gt_len={len(row['gt'])}")

0_tlyarata_6-7_na_pechat_page_005_block_012.png: 377x335, ratio=1.1, gt_len=80
0_tlyarata_6-7_na_pechat_page_001_block_000.png: 556x217, ratio=2.6, gt_len=52
0_tlyarata_8_-na_pechat_page_003_block_016.png: 376x424, ratio=0.9, gt_len=80
0_tlyarata_8_-na_pechat_page_003_block_021.png: 376x561, ratio=0.7, gt_len=80
0_tlyarata_6-7_na_pechat_page_003_block_031.png: 376x771, ratio=0.5, gt_len=80
0_tlyarata_6-7_na_pechat_page_005_block_003.png: 375x473, ratio=0.8, gt_len=80
0_tlyarata_6-7_na_pechat_page_004_block_012.png: 376x1005, ratio=0.4, gt_len=80
0_tlyarata_8_-na_pechat_page_003_block_019.png: 376x1094, ratio=0.3, gt_len=80
0_tlyarata_6-7_na_pechat_page_003_block_017.png: 377x357, ratio=1.1, gt_len=80
0_tlyarata_6-7_na_pechat_page_001_block_009.png: 383x473, ratio=0.8, gt_len=80
0_tlyarata_6-7_na_pechat_page_002_block_003.png: 765x783, ratio=1.0, gt_len=80
0_tlyarata_6-7_na_pechat_page_002_block_007.png: 289x252, ratio=1.1, gt_len=80
0_tlyarata_6-7_na_pechat_page_003_block_024.png: 375x

### Качество дообученной модели

In [28]:
rows = []
for item in items: # из ячейки инференса
    rows.append({
        'file': item['file'],
        'pred': item['pred'],
        'gt': item['gt'],
        'cer': cer(item['pred'], item['gt']),
        'wer': wer(item['pred'], item['gt']),
        'match': item['pred'] == item['gt'],
    })

eval_df = pd.DataFrame(rows)


print(f"Всего примеров: {len(eval_df)}")
print(f"Mean CER: {eval_df['cer'].mean():.4f}")
print(f"Mean WER: {eval_df['wer'].mean():.4f} ({eval_df['wer'].mean()*100:.1f}%)")
print(f"Exact match: {eval_df['match'].mean():.4f} ({eval_df['match'].mean()*100:.1f}%)")



Всего примеров: 29
Mean CER: 0.6360
Mean WER: 0.7644 (76.4%)
Exact match: 0.2414 (24.1%)


## 12. Notes

- **Только посмотреть модель:** §2 → §3 → §11 (§1 не нужна).
- **Обучить заново:** §1 + restart → §2 … §10.

## Структура папки

In [27]:
import os

def list_files(startpath, max_depth=3):
    """Рекурсивный обход папок с ограничением глубины"""

    def _list_files(current_path, depth=0):
        if depth > max_depth:
            return

        try:
            items = os.listdir(current_path)
            indent = "  " * depth

            for item in sorted(items):  # сортировка для удобства
                full_path = os.path.join(current_path, item)

                if os.path.isdir(full_path):
                    print(f"{indent}📁 {item}/")
                    _list_files(full_path, depth + 1)
                else:
                    print(f"{indent}📄 {item}")

        except PermissionError:
            print(f"{indent}❌ Нет доступа к {current_path}")
        except FileNotFoundError:
            print(f"{indent}❌ Папка не найдена: {current_path}")

    print(f"📁 Структура {startpath}")
    print("=" * 50)
    _list_files(startpath)

# Использование
path = '/content/drive/MyDrive/Avar OCR ANN/'
list_files(path, max_depth=3)

📁 Структура /content/drive/MyDrive/Avar OCR ANN/
📄 README.md
📄 baseline_ocr_on_classical_crops.ipynb
📁 classical_layout/
  📁 0_tlyarata_6-7_na_pechat_page_001/
    📁 connected_components/
      📁 crops/
      📄 debug_connected_components.png
      📄 manual_corrections.md
      📄 metadata.json
      📄 ocr_results.json
      📄 ocr_text.md
      📄 ocr_text.txt
  📁 0_tlyarata_6-7_na_pechat_page_002/
    📁 connected_components/
      📁 crops/
      📄 debug_connected_components.png
      📄 manual_corrections.md
      📄 metadata.json
      📄 ocr_results.json
      📄 ocr_text.md
      📄 ocr_text.txt
  📁 0_tlyarata_6-7_na_pechat_page_003/
    📁 connected_components/
      📁 crops/
      📄 debug_connected_components.png
      📄 manual_corrections.md
      📄 metadata.json
      📄 ocr_results.json
      📄 ocr_text.md
      📄 ocr_text.txt
  📁 0_tlyarata_6-7_na_pechat_page_004/
    📁 connected_components/
      📁 crops/
      📄 debug_connected_components.png
      📄 manual_corrections.md
      📄 met